In [28]:
import pandas as pd
import numpy as np

In [29]:
df = pd.read_csv("mail_data.csv")

In [30]:
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [31]:
df.isnull().sum()

Category    0
Message     0
dtype: int64

In [32]:
df.duplicated().sum()

np.int64(415)

In [33]:
df = df.drop_duplicates()

In [34]:
df.duplicated().sum()

np.int64(0)

In [35]:
x = df["Message"]

In [36]:
y = df["Category"]

In [37]:
from sklearn.preprocessing import LabelEncoder

In [38]:
le = LabelEncoder()

In [39]:
y = le.fit_transform(y)

In [40]:
from sklearn.model_selection import train_test_split

In [41]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [53]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


def preprocess(text):
    text = text.lower()
    text = re.sub("[^a-zA-Z0-9 ]","",text)
    text = text.split()
    text = [word for word in text if word not in ENGLISH_STOP_WORDS]
    text = " ".join(text)
    return text

In [54]:
def transform_text(x):
    return [preprocess(text) for text in x]

In [55]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer 
from sklearn.feature_extraction.text import TfidfVectorizer

In [56]:
preprocessing = Pipeline([
    ("preprocess",FunctionTransformer(transform_text,validate=False)),
    ("tfidf",TfidfVectorizer())
])

In [57]:
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB

In [58]:
model = Pipeline([
    ("preprocessing",preprocessing),
    ("nb",MultinomialNB())
])

In [59]:
model.fit(x_train,y_train)

,steps,"[('preprocessing', ...), ('nb', ...)]"
,transform_input,None
,memory,None
,verbose,False
,steps,"[('preprocess', ...), ('tfidf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,func,<function tra...0016D63FE8AE0>
,inverse_func,None
,validate,False


In [60]:
model.predict(x_test)

array([0, 0, 0, ..., 0, 0, 0], shape=(1032,))

In [61]:
model.predict(["Congratulations! You have won a free iPhone."])

array([1])

In [62]:
email = pd.Series(["Congratulations! You won a free iPhone"])

prediction = model.predict(email)
prediction

array([1])

In [63]:
import pickle

In [64]:
pickle.dump(model, open("model.pkl", "wb"))